📌 Cell 1: Install Required Libraries

In [2]:
!pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cpu
!pip install transformers datasets sentencepiece accelerate tqdm


Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 16.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 4.0 MB/s eta 0:00:0000:0100:010m


📌 Cell 2: Import Libraries and Set Device (MPS Support)

In [3]:
import torch
from datasets import load_dataset
from transformers import T5ForConditionalGeneration, T5Tokenizer, Trainer, TrainingArguments

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("✅ Using device:", device)

✅ Using device: mps


📌 Cell 3: Load Dataset (Small Subset First)

In [9]:
import pandas as pd
from datasets import Dataset

# Path to your shard
tsv_path = "./data/C4_200M.tsv-00000-of-00010"

# Load using pandas (for performance, limit rows during testing)
df = pd.read_csv(tsv_path, sep="\t", names=["original", "corrected"], nrows=100000)
df = df.dropna()

# Convert to Hugging Face Dataset
dataset = Dataset.from_pandas(df)
dataset


Dataset({
    features: ['original', 'corrected'],
    num_rows: 100000
})

📌 Cell 4: Tokenizer & Preprocessing for T5

In [11]:
tokenizer = T5Tokenizer.from_pretrained("t5-base")

def preprocess_function(batch):
    inputs = ["grammar: " + item for item in batch["original"]]
    targets = batch["corrected"]

    model_inputs = tokenizer(inputs, max_length=128, padding="max_length", truncation=True)
    labels = tokenizer(targets, max_length=128, padding="max_length", truncation=True)
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

tokenized_dataset = dataset.map(preprocess_function, batched=True)
tokenized_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])


Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

📌 Cell 5: Load Model & Training Arguments

In [13]:
model = T5ForConditionalGeneration.from_pretrained("t5-base").to(device)

training_args = TrainingArguments(
    output_dir="./t5_c4_model",
    evaluation_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100
)


📌 Cell 6: Trainer Setup

In [14]:
from transformers import DataCollatorForSeq2Seq

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset.select(range(3500)), 
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer)
)


/var/folders/c4/1kx94fgd5pj0cm10pvy6kwg40000gn/T/ipykernel_79199/1623194537.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


📌 Cell 7: Train the Model

In [15]:
trainer.train()

/opt/anaconda3/lib/python3.11/site-packages/transformers/data/data_collator.py:740: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:257.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

📌 Cell 8: Save the Model

In [ ]:
model.save_pretrained("./t5_c4_grammar_corrector")
tokenizer.save_pretrained("./t5_c4_grammar_corrector")

📌 Cell 9: Inference Function

In [ ]:
def correct_grammar(sentence):
    input_text = "grammar: " + sentence
    input_ids = tokenizer.encode(input_text, return_tensors="pt").to(device)
    outputs = model.generate(input_ids, max_length=128, num_beams=4, early_stopping=True)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test
test_sentence = "She dont goes to the school yesterday."
print("Original:", test_sentence)
print("Corrected:", correct_grammar(test_sentence))
